# UPI Transactions Data Analysis

An end-to-end, question-driven analysis of 20,000 UPI transactions.

The notebook is deliberately thin: every computation lives in the reusable `src/` package
(the same code the Streamlit dashboard and CLI use), so results here always match the
dashboard and the Power BI model.

**Guiding questions**
1. How much value flows through the network, and how healthy is it?
2. Which banks, cities and merchants drive the volume?
3. Is activity seasonal, and when do people transact?
4. Where are transactions failing?
5. Who are the customers (age, gender, device, payment method)?

## 0. Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / 'notebooks').exists() or Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config, data_cleaning, data_loader, feature_engineering, analysis, visualization

pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
pd.set_option('display.max_columns', 40)
print('Project root:', PROJECT_ROOT)

Project root: D:\portfolio\UPI Transactions Data Analysis


## 1. Load the raw workbook

In [2]:
raw = data_loader.validate_schema(data_loader.load_raw_data())
print(f'{raw.shape[0]:,} rows x {raw.shape[1]} columns')
raw.head()

20,000 rows x 20 columns


,TransactionID,TransactionDate,Amount,BankNameSent,BankNameReceived,RemainingBalance,City,Gender,TransactionType,Status,TransactionTime,DeviceType,PaymentMethod,MerchantName,Purpose,CustomerAge,PaymentMode,Currency,CustomerAccountNumber,MerchantAccountNumber
0,TXN00001,2024-02-02,271.64,SBI Bank,HDFC Bank,"5,557.02",Delhi,Female,Transfer,Success,17:12:14,Tablet,Phone Number,Amazon,Food,21,Scheduled,USD,123456789013,987654321013
1,TXN00002,2024-03-03,"1,064.63",ICICI Bank,SBI Bank,"9,753.32",Bangalore,Male,Payment,Success,11:15:02,Laptop,QR Code,Zomato,Travel,22,Instant,EUR,123456789014,987654321014
2,TXN00003,2024-04-04,144.15,Axis Bank,Axis Bank,"7,597.35",Hyderabad,Female,Transfer,Success,21:29:39,Mobile,UPI ID,Swiggy,Bill Payment,23,Scheduled,GBP,123456789015,987654321015
3,TXN00004,2024-05-05,612.89,HDFC Bank,ICICI Bank,"2,327.84",Mumbai,Male,Payment,Success,06:27:36,Tablet,Phone Number,IRCTC,Others,24,Instant,INR,123456789016,987654321016
4,TXN00005,2024-06-06,743.32,SBI Bank,HDFC Bank,"1,136.84",Delhi,Female,Transfer,Failed,02:06:22,Laptop,QR Code,Flipkart,Shopping,25,Scheduled,USD,123456789017,987654321017


In [3]:
profile = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing': raw.isna().sum(),
    'unique': raw.nunique(),
})
profile

,dtype,missing,unique
TransactionID,str,0,20000
TransactionDate,datetime64[us],0,60
Amount,float64,0,19054
BankNameSent,str,0,4
BankNameReceived,str,0,4
RemainingBalance,float64,0,19781
City,str,0,4
Gender,str,0,2
TransactionType,str,0,2
Status,str,0,2


## 2. Clean and enrich

In [4]:
clean = data_cleaning.clean_transactions(raw)
data_cleaning.cleaning_report(raw, clean)

{'rows_before': 20000,
 'rows_after': 20000,
 'rows_removed': 0,
 'duplicates_removed': 0,
 'missing_values_after': 0,
 'numeric_amounts': True,
 'amount_is_finite': True}

In [5]:
df = feature_engineering.add_features(clean)
df[['TransactionID', 'YearMonth', 'Hour', 'DayPart', 'AgeGroup', 'AmountBand',
    'IsSuccess', 'EstimatedRevenue', 'SettledAmount']].head()

,TransactionID,YearMonth,Hour,DayPart,AgeGroup,AmountBand,IsSuccess,EstimatedRevenue,SettledAmount
0,TXN00001,2024-02,17,Evening,20-25,250-500,1,0.00,271.64
1,TXN00002,2024-03,11,Morning,20-25,1000-1500,1,9.58,"1,064.63"
2,TXN00003,2024-04,21,Evening,20-25,<250,1,0.00,144.15
3,TXN00004,2024-05,6,Morning,20-25,500-1000,1,5.52,612.89
4,TXN00005,2024-06,2,Night,20-25,500-1000,0,0.00,0.00


## Q1 — How much value flows through the network?

In [6]:
kpis = analysis.overall_kpis(df)
for key, value in kpis.items():
    print(f'{key:>24}: {value}')

      total_transactions: 20000
 successful_transactions: 16000
     failed_transactions: 4000
        success_rate_pct: 80.0
             total_value: 19872274.03
           settled_value: 15875486.95
       estimated_revenue: 71290.75
          average_amount: 993.61
        unique_customers: 20000
        unique_merchants: 5
           unique_cities: 4
              date_start: 2024-01-01
                date_end: 2024-12-30


## Q2 — Which sending banks handle the most value?

In [7]:
analysis.transactions_by_bank(df)

,bank,transactions,total_value,average_amount,success_rate,estimated_revenue,share_pct
0,HDFC Bank,5000,"5,052,564.92","1,010.51",80.00,"36,371.35",25.43
1,SBI Bank,5000,"5,036,738.71","1,007.35",80.00,0.00,25.35
2,Axis Bank,5000,"4,907,634.17",981.53,80.00,0.00,24.70
3,ICICI Bank,5000,"4,875,336.23",975.07,80.00,"34,919.41",24.53


In [8]:
visualization.plot_top_banks(df);

## Q3 — Where is the volume coming from (cities)?

In [9]:
analysis.transactions_by_city(df)

,city,transactions,total_value,average_amount,success_rate,estimated_revenue,share_pct
0,Mumbai,5000,"5,052,564.92","1,010.51",80.00,"36,371.35",25.43
1,Delhi,5000,"5,036,738.71","1,007.35",80.00,0.00,25.35
2,Hyderabad,5000,"4,907,634.17",981.53,80.00,0.00,24.70
3,Bangalore,5000,"4,875,336.23",975.07,80.00,"34,919.41",24.53


In [10]:
visualization.plot_city_distribution(df);

## Q4 — Is activity seasonal?

In [11]:
analysis.monthly_trend(df)

,YearMonth,transactions,total_value,settled_value,failed_value,estimated_revenue,success_rate,mom_growth_pct
0,2024-01,1666,"1,678,734.80","1,348,413.39","330,321.41","12,135.72",80.01,NaN
1,2024-02,1667,"1,692,737.83","1,346,273.79","346,464.04",0.00,80.02,0.83
2,2024-03,1667,"1,623,693.14","1,284,136.90","339,556.24","11,557.23",80.02,-4.08
3,2024-04,1667,"1,662,906.93","1,330,767.29","332,139.64",0.00,79.96,2.42
4,2024-05,1667,"1,706,785.86","1,375,444.57","331,341.29","12,379.00",80.02,2.64
5,2024-06,1667,"1,652,588.38","1,326,142.70","326,445.68",0.00,79.96,-3.18
6,2024-07,1667,"1,609,833.25","1,277,148.40","332,684.85","11,494.33",80.02,-2.59
7,2024-08,1667,"1,598,708.63","1,284,585.18","314,123.45",0.00,80.02,-0.69
8,2024-09,1667,"1,667,044.26","1,317,403.59","349,640.67","11,856.63",79.96,4.27
9,2024-10,1666,"1,691,412.50","1,346,728.57","344,683.93",0.00,80.01,1.46


In [12]:
visualization.plot_monthly_trend(df);

In [13]:
visualization.plot_day_of_week(df);

## Q5 — Where are transactions failing?

In [14]:
print(analysis.status_breakdown(df))
visualization.plot_status_breakdown(df);

    Status  transactions   total_value  share_pct
0   Failed          4000  3,996,787.08      20.00
1  Success         16000 15,875,486.95      80.00


In [15]:
analysis.success_rate_by_dimension(df, 'BankNameSent')
visualization.plot_success_rate_by_bank(df);

## Q6 — Which devices and payment methods are used?

In [16]:
display(analysis.transactions_by_device(df))
visualization.plot_device_breakdown(df);

,device,transactions,total_value,average_amount,success_rate,estimated_revenue,share_pct
0,Mobile,6666,"6,642,887.48",996.53,80.00,"23,630.05",33.43
1,Tablet,6667,"6,640,042.16",995.96,80.01,"24,246.84",33.41
2,Laptop,6667,"6,589,344.39",988.35,79.99,"23,413.86",33.16


In [17]:
display(analysis.transactions_by_payment_method(df))
visualization.plot_payment_method(df);

,payment_method,transactions,total_value,average_amount,success_rate,estimated_revenue,share_pct
0,UPI ID,6666,"6,642,887.48",996.53,80.00,"23,630.05",33.43
1,Phone Number,6667,"6,640,042.16",995.96,80.01,"24,246.84",33.41
2,QR Code,6667,"6,589,344.39",988.35,79.99,"23,413.86",33.16


## Q7 — Which merchants and purposes dominate?

In [18]:
display(analysis.transactions_by_merchant(df))
visualization.plot_merchant_performance(df);

,merchant,transactions,total_value,average_amount,success_rate,estimated_revenue,share_pct
0,Zomato,4000,"3,998,155.96",999.54,100.00,"18,082.58",20.12
1,Flipkart,4000,"3,996,787.08",999.20,0.00,0.00,20.11
2,IRCTC,4000,"3,986,338.75",996.58,100.00,"17,795.45",20.06
3,Swiggy,4000,"3,945,853.73",986.46,100.00,"17,803.83",19.86
4,Amazon,4000,"3,945,138.51",986.28,100.00,"17,608.89",19.85


In [19]:
display(analysis.transactions_by_purpose(df))
visualization.plot_purpose_breakdown(df);

,purpose,transactions,total_value,average_amount,success_rate,estimated_revenue,share_pct
0,Travel,4000,"3,998,155.96",999.54,100.00,"18,082.58",20.12
1,Shopping,4000,"3,996,787.08",999.20,0.00,0.00,20.11
2,Others,4000,"3,986,338.75",996.58,100.00,"17,795.45",20.06
3,Bill Payment,4000,"3,945,853.73",986.46,100.00,"17,803.83",19.86
4,Food,4000,"3,945,138.51",986.28,100.00,"17,608.89",19.85


## Q8 — Who are the customers?

In [20]:
display(analysis.transactions_by_age_group(df))
visualization.plot_age_groups(df);

,AgeGroup,transactions,total_value,average_amount,success_rate,estimated_revenue,share_pct
0,20-25,3000,"2,946,355.93",982.12,66.67,"8,841.63",14.83
1,26-35,5000,"4,989,280.09",997.86,80.00,"17,831.43",25.11
2,36-45,5000,"5,003,482.25","1,000.70",80.00,"17,882.29",25.18
3,46-55,5000,"4,940,523.78",988.10,80.00,"17,713.22",24.86
4,56+,2000,"1,992,631.98",996.32,100.00,"9,022.18",10.03


In [21]:
analysis.transactions_by_gender(df)

,gender,transactions,total_value,average_amount,success_rate,estimated_revenue,share_pct
0,Female,10000,"9,944,372.88",994.44,80.00,0.00,50.04
1,Male,10000,"9,927,901.15",992.79,80.00,"71,290.75",49.96


## Q9 — When do people transact?

In [22]:
analysis.peak_hours(df).sort_values('transactions', ascending=False).head(5)

,Hour,transactions,total_value
9,9,904,"899,599.26"
0,0,887,"845,290.83"
2,2,870,"881,487.59"
10,10,865,"861,478.25"
20,20,857,"841,704.81"


In [23]:
visualization.plot_peak_hours(df);

## Q10 — How do the numeric drivers relate?

In [24]:
analysis.correlation_matrix(df)
visualization.plot_correlation_heatmap(df);

## Data quality caveats

This is **synthetic** data with several near-perfectly confounded columns. Treat the correlations below as artefacts of how the sample was generated rather than business insight:

- BankNameSent determines TransactionType (SBI/Axis -> Transfer only, HDFC/ICICI -> Payment only), so estimated revenue concentrates in two banks.
- MerchantName maps 1:1 to Purpose (e.g. Flipkart <-> Shopping).
- Flipkart shows a 0% success rate and Gender aligns with the bank/type split - both spurious.
- Status swings with specific calendar dates and weekday.

Spotting these is part of the analysis: verify confounding before drawing conclusions.

## Key findings

In [25]:
for line in analysis.insight_highlights(df):
    print('•', line)

• 20,000 transactions worth 19,872,274.03 processed between 2024-01-01 and 2024-12-30.
• Overall success rate is 80.0%; 4,000 transactions failed.
• HDFC Bank is the busiest sending bank and Mumbai the top city.
• Mobiles are the preferred device; activity peaks around 09:00, with Mon the most active day.
• Estimated platform revenue is 71,290.75 (assumption: 0.90% MDR on successful payments).


## Next step — Power BI

Run the pipeline once to export the star schema:

```bash
python -m src.run_analysis
```

Then follow `powerbi/build_guide.md` to assemble the report from the CSVs in `powerbi/`.
The DAX measures are documented in `powerbi/dax_measures.md`.